In [0]:
from pyspark.sql import functions as F

patients_bronze = spark.table("bronze_patients")

display(patients_bronze)

patient_id,first_name,last_name,date_of_birth,gender,blood_group,email,phone,created_at,updated_at
1,Aarav,Singh,1963-10-22,Male,B+,patient1@example.com,917000000000,2026-03-05T15:57:18.000Z,2024-01-08T13:19:29.000Z
2,Kabir,Patel,1999-01-22,Female,B-,patient2@example.com,917000000001,2025-05-30T16:44:19.000Z,2025-07-04T16:16:32.000Z
3,Aditya,Singh,1985-06-25,Male,AB+,patient3@example.com,917000000002,2024-11-06T19:21:00.000Z,2025-04-02T20:14:27.000Z
4,Rohan,Sharma,1969-09-25,Female,A-,patient4@example.com,917000000003,2025-05-09T16:22:39.000Z,2025-03-24T06:39:54.000Z
5,Rohan,Singh,1957-04-13,Male,B+,patient5@example.com,917000000004,2026-07-19T05:42:51.000Z,2025-10-15T11:07:49.000Z
6,Kabir,Singh,2002-12-31,Female,A-,patient6@example.com,917000000005,2026-04-04T23:37:32.000Z,2025-05-17T18:31:20.000Z
7,Aarav,Singh,1957-02-16,Female,O-,patient7@example.com,917000000006,2026-03-13T00:25:52.000Z,2025-12-01T02:15:47.000Z
8,Aditya,Nair,1990-08-15,Female,AB-,patient8@example.com,917000000007,2025-03-21T05:06:42.000Z,2026-01-04T21:27:47.000Z
9,Arjun,Reddy,1956-12-21,Male,O+,patient9@example.com,917000000008,2026-01-24T08:10:28.000Z,2026-06-04T03:40:42.000Z
10,Aarav,Nair,1986-08-07,Female,O+,patient10@example.com,917000000009,2025-01-05T10:56:46.000Z,2026-02-07T20:57:43.000Z


In [0]:
patients_silver = (
    patients_bronze
    .dropDuplicates(["patient_id"])
    .withColumn("first_name", F.trim(F.initcap(F.col("first_name"))))
    .withColumn("last_name", F.trim(F.initcap(F.col("last_name"))))
    .withColumn("gender", F.upper(F.trim(F.col("gender"))))
    .withColumn("blood_group", F.upper(F.trim(F.col("blood_group"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("phone", F.trim(F.col("phone")))
)

In [0]:
# Check required fields for NULL values

required_columns = [
    "patient_id",
    "first_name",
    "last_name",
    "date_of_birth",
    "gender"
]

for column in required_columns:
    null_count = patients_silver.filter(
        F.col(column).isNull()
    ).count()

    print(f"{column}: {null_count} NULL values")

patient_id: 0 NULL values
first_name: 0 NULL values
last_name: 0 NULL values
date_of_birth: 0 NULL values
gender: 0 NULL values


In [0]:
# Validate patient date of birth

invalid_dob = patients_silver.filter(
    (F.col("date_of_birth") > F.current_date()) |
    (F.col("date_of_birth") < F.lit("1900-01-01"))
)

print("Invalid date-of-birth records:", invalid_dob.count())

display(invalid_dob)

Invalid date-of-birth records: 0


patient_id,first_name,last_name,date_of_birth,gender,blood_group,email,phone,created_at,updated_at


In [0]:
patients_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_patients")

In [0]:
display(spark.table("silver_patients"))

patient_id,first_name,last_name,date_of_birth,gender,blood_group,email,phone,created_at,updated_at
12,Karan,Singh,1969-03-17,MALE,A+,patient12@example.com,917000000011,2025-09-07T06:14:12.000Z,2024-08-14T18:02:23.000Z
18,Arjun,Patel,1998-09-02,FEMALE,A-,patient18@example.com,917000000017,2024-06-07T20:18:26.000Z,2026-01-26T02:08:57.000Z
38,Kabir,Sharma,1980-10-10,MALE,A+,patient38@example.com,917000000037,2026-04-01T17:33:12.000Z,2026-02-04T00:06:26.000Z
67,Karan,Gupta,1978-05-30,FEMALE,O-,patient67@example.com,917000000066,2025-09-04T01:01:40.000Z,2026-07-04T01:10:56.000Z
70,Rohan,Sharma,1949-11-05,MALE,A+,patient70@example.com,917000000069,2025-11-29T03:19:20.000Z,2024-09-17T16:05:25.000Z
93,Kabir,Iyer,1995-03-28,FEMALE,AB+,patient93@example.com,917000000092,2024-02-18T05:46:29.000Z,2026-06-02T12:59:11.000Z
16,Kabir,Iyer,1946-04-07,FEMALE,AB+,patient16@example.com,917000000015,2025-02-21T03:23:14.000Z,2026-03-23T22:22:54.000Z
64,Karan,Reddy,1969-01-21,FEMALE,A-,patient64@example.com,917000000063,2026-06-03T00:29:19.000Z,2024-01-05T16:54:49.000Z
74,Arjun,Nair,1970-05-13,FEMALE,AB+,patient74@example.com,917000000073,2026-06-19T08:58:43.000Z,2024-04-27T06:55:39.000Z
94,Rohan,Gupta,1968-07-12,FEMALE,O+,patient94@example.com,917000000093,2024-04-01T10:29:45.000Z,2025-12-11T04:09:48.000Z


In [0]:
appointments_bronze = spark.table("bronze_appointments")

appointments_silver = (
    appointments_bronze
    .dropDuplicates(["appointment_id"])
    .withColumn("appointment_status",
                F.upper(F.trim(F.col("appointment_status"))))
    .withColumn("visit_type",
                F.upper(F.trim(F.col("visit_type"))))
    .withColumn("reason",
                F.trim(F.col("reason")))
)

In [0]:
required_columns = [
    "appointment_id",
    "patient_id",
    "doctor_id",
    "department_id",
    "appointment_date"
]

for column in required_columns:
    null_count = appointments_silver.filter(
        F.col(column).isNull()
    ).count()

    print(f"{column}: {null_count} NULL values")

appointment_id: 0 NULL values
patient_id: 0 NULL values
doctor_id: 0 NULL values
department_id: 0 NULL values
appointment_date: 0 NULL values


In [0]:
# Referential integrity checks for appointments

patients_ids = spark.table("silver_patients").select("patient_id").distinct()
doctors_ids = spark.table("bronze_doctors").select("doctor_id").distinct()
departments_ids = spark.table("bronze_departments").select("department_id").distinct()

invalid_patients = appointments_silver.join(
    patients_ids,
    "patient_id",
    "left_anti"
)

invalid_doctors = appointments_silver.join(
    doctors_ids,
    "doctor_id",
    "left_anti"
)

invalid_departments = appointments_silver.join(
    departments_ids,
    "department_id",
    "left_anti"
)

print("Invalid patient references:", invalid_patients.count())
print("Invalid doctor references:", invalid_doctors.count())
print("Invalid department references:", invalid_departments.count())

Invalid patient references: 0
Invalid doctor references: 0
Invalid department references: 0


In [0]:
appointments_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_appointments")

In [0]:
display(spark.table("silver_appointments"))

appointment_id,patient_id,doctor_id,department_id,appointment_date,appointment_status,visit_type,reason,wait_time_minutes,consultation_duration_minutes,created_at,updated_at
12,12,1,1,2024-09-20,SCHEDULED,SPECIALIST,Follow-up,37,19,2026-04-24T15:03:12.000Z,2025-04-27T22:12:52.000Z
18,18,1,1,2026-06-26,COMPLETED,EMERGENCY,Acute Symptoms,9,12,2024-08-30T03:29:43.000Z,2025-05-30T06:47:12.000Z
38,38,1,1,2026-06-01,COMPLETED,FOLLOW-UP,Routine Checkup,77,19,2026-01-26T21:19:53.000Z,2024-04-13T07:14:23.000Z
67,67,1,1,2025-05-22,SCHEDULED,SPECIALIST,Follow-up,82,50,2026-08-05T04:36:54.000Z,2025-04-13T12:54:22.000Z
70,70,1,1,2024-02-14,COMPLETED,EMERGENCY,Acute Symptoms,79,18,2025-12-20T02:27:31.000Z,2026-01-02T04:15:23.000Z
93,93,1,1,2024-08-25,COMPLETED,SPECIALIST,Routine Checkup,68,47,2024-11-19T21:03:20.000Z,2025-07-24T02:01:49.000Z
161,61,1,1,2024-12-01,SCHEDULED,SPECIALIST,Acute Symptoms,57,27,2024-09-29T19:23:41.000Z,2026-05-22T10:33:46.000Z
186,86,1,1,2025-01-28,COMPLETED,FOLLOW-UP,Chronic Disease Review,74,38,2026-04-03T10:27:44.000Z,2024-01-14T22:39:53.000Z
190,90,1,1,2026-02-04,COMPLETED,SPECIALIST,Chronic Disease Review,51,24,2024-05-04T05:29:58.000Z,2024-07-14T18:12:52.000Z
218,18,1,1,2024-05-03,COMPLETED,SPECIALIST,Acute Symptoms,14,25,2025-10-17T11:31:59.000Z,2024-05-25T10:15:05.000Z


In [0]:
tables_to_clean = [
    "doctors",
    "departments",
    "diagnoses",
    "treatments",
    "medications",
    "insurance_providers",
    "patient_addresses",
    "patient_contacts",
    "patient_insurance",
    "doctor_departments",
    "appointment_diagnoses",
    "appointment_treatments",
    "prescriptions",
    "prescription_items",
    "lab_tests",
    "lab_test_results",
    "billing",
    "insurance_claims",
    "payments"
]

silver_dataframes = {}

for table in tables_to_clean:
    df = spark.table(f"bronze_{table}")
    
    # Remove exact duplicate records
    df = df.dropDuplicates()
    
    # Trim whitespace from all string columns
    for column, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(column, F.trim(F.col(column)))
    
    silver_dataframes[table] = df
    
    print(f"{table}: {df.count()} records after basic cleansing")

doctors: 1 records after basic cleansing
departments: 1 records after basic cleansing
diagnoses: 1 records after basic cleansing
treatments: 1 records after basic cleansing
medications: 1 records after basic cleansing
insurance_providers: 1 records after basic cleansing
patient_addresses: 120 records after basic cleansing
patient_contacts: 150 records after basic cleansing
patient_insurance: 120 records after basic cleansing
doctor_departments: 1 records after basic cleansing
appointment_diagnoses: 600 records after basic cleansing
appointment_treatments: 600 records after basic cleansing
prescriptions: 300 records after basic cleansing
prescription_items: 600 records after basic cleansing
lab_tests: 400 records after basic cleansing
lab_test_results: 400 records after basic cleansing
billing: 500 records after basic cleansing
insurance_claims: 200 records after basic cleansing
payments: 300 records after basic cleansing


In [0]:
for table, df in silver_dataframes.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"silver_{table}")
    )
    
    print(f"Created: silver_{table}")

Created: silver_doctors
Created: silver_departments
Created: silver_diagnoses
Created: silver_treatments
Created: silver_medications
Created: silver_insurance_providers
Created: silver_patient_addresses
Created: silver_patient_contacts
Created: silver_patient_insurance
Created: silver_doctor_departments
Created: silver_appointment_diagnoses
Created: silver_appointment_treatments
Created: silver_prescriptions
Created: silver_prescription_items
Created: silver_lab_tests
Created: silver_lab_test_results
Created: silver_billing
Created: silver_insurance_claims
Created: silver_payments
